In [1]:
%load_ext rpy2.ipython

In [2]:
import pandas as pd

import src
import src.load

r_colormap = src.r_colormap
pd.options.display.float_format = "{:.1f}".format

In [3]:
%%R -i r_colormap

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

here() starts at /mnt/nvme_storage/git/ytpop


In [19]:
# Load all the Data

channel_df = src.load.channels()
video_df = src.load.videos(filter_period=True, filter_sentences=False)
sentence_df = src.load.sentences(filter_video=False)
popbert_df = src.load.popbert()
comments_df = src.load.comments()


videos = pd.merge(
    channel_df,
    video_df,
    on="channel_id",
    how="inner",
)

sents = sentence_df.merge(popbert_df, on="sentence_id")
sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# per Channel (all videos)

In [20]:
channel_overview = (
    videos.merge(sents, on="video_id")
    .groupby("channel", observed=True)
    .agg(
        ch_videos=("channel", "size"),
        ch_followers=("channel_followers", "first"),
        n_sentences=("n_sents", "sum"),
        n_elite=("n_elite", "sum"),
        n_pplcentr=("n_pplcentr", "sum"),
        avg_likes=("video_likes", "mean"),
        avg_views=("video_views", "mean"),
        avg_duration=("video_duration", "mean"),
        avg_comments=("video_comments", "mean"),
        first_video=("video_uploadtime", "min"),
        latest_video=("video_uploadtime", "max"),
    )
)

In [17]:
channel_overview

,videos,follower_count,n_sentences,n_elite,n_pplcentr,avg_likes,avg_views,avg_duration,avg_comments,first_video,latest_video
channel,,,,,,,,,,,
AfD TV,1454,250000,142870,17059,3595,3636.7,43377.5,661.0,397.2,2017-12-07,2024-01-19
AfD BT,5220,388000,295280,39749,5985,3859.2,45732.7,436.1,397.4,2017-12-06,2024-01-20
Greens,457,26100,45677,1415,1314,79.6,4512.6,897.6,0.2,2018-01-27,2023-12-13
CDU,632,21900,53112,919,1429,66.5,9282.0,643.2,42.6,2017-12-11,2024-01-19
CSU,145,5170,10357,286,249,35.6,22332.8,442.2,7.5,2017-12-14,2023-10-05
Left,435,29000,45942,2421,1375,257.5,10604.9,869.7,46.1,2017-12-11,2024-01-17
FDP,483,23300,36204,1406,893,0.5,5441.0,641.8,0.7,2018-01-06,2024-01-06
SPD,477,24200,65563,1383,2163,102.1,5168.7,1114.7,25.6,2017-12-07,2024-01-18


In [21]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 1489.35 hours


In [22]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 9628


In [25]:
# number of sentencs

count_sents = channel_overview.n_sentences.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 695005


In [10]:
comments_query = (
    Query(Comment.id, func.count(Comment.id))
    .filter(Comment.video_id == Video.id, Comment.is_valid == True)
    .with_entities(func.count(Comment.id))
    .scalar_subquery()
)
query = (
    Query(Video)
    .join(Channel)
    .filter(Video.is_valid == True)
    .with_entities(
        Channel.channel,
        Video.datetime_upload,
        Video.like_count.label("likes"),
        Video.view_count.label("views"),
        Video.duration.label("duration"),
        comments_query.label("comments"),
    )
)

with engine.connect() as conn:
    videos = pd.read_sql(query.statement, conn)
videos.channel = videos.channel.replace(party_names)

In [27]:
channel_overview.drop(["first_video", "latest_video"], axis=1).T

channel,AfD TV,AfD BT,Greens,CDU,CSU,Left,FDP,SPD
videos,1454.0,5220.0,457.0,632.0,145.0,435.0,483.0,477.0
follower_count,250000.0,388000.0,26100.0,21900.0,5170.0,29000.0,23300.0,24200.0
n_sentences,142870.0,295280.0,45677.0,53112.0,10357.0,45942.0,36204.0,65563.0
n_elite,17059.0,39749.0,1415.0,919.0,286.0,2421.0,1406.0,1383.0
n_pplcentr,3595.0,5985.0,1314.0,1429.0,249.0,1375.0,893.0,2163.0
avg_likes,3636.7,3859.2,79.6,66.5,35.6,257.5,0.5,102.1
avg_views,43377.5,45732.7,4512.6,9282.0,22332.8,10604.9,5441.0,5168.7
avg_duration,661.0,436.1,897.6,643.2,442.2,869.7,641.8,1114.7
avg_comments,397.2,397.4,0.2,42.6,7.5,46.1,0.7,25.6


In [33]:
summary_table = pd.merge(df_valid, pivot_videos, on="channel").T
summary_table.columns = [col.lstrip("@") for col in summary_table.iloc[0, :]]
summary_table = summary_table.iloc[1:, :]
summary_table = summary_table

In [34]:
summary_table

,AfD BT,AfD TV,Left,Greens,SPD,FDP,CDU,CSU
ch_followers,388000,250000,29000,26100,24200,23300,21900,5170
ch_videos,5207,1457,435,459,480,487,629,145
n_sentences,299414,145525,46823,46964,66999,36930,54267,10463
n_sent_elite,40413,17477,2499,1463,1433,1400,947,290
n_sent_pplcentr,6117,3663,1425,1358,2253,929,1450,239
avg_comments,398.3,398.7,45.0,0.2,25.0,0.7,41.2,7.5
avg_duration,436.2,657.2,832.4,892.2,1088.2,620.3,616.0,442.2
avg_likes,3867.4,3627.1,255.8,79.7,102.1,0.5,64.4,35.6
avg_views,45832.0,43363.7,10426.9,4537.7,5065.4,5874.0,9624.1,22332.8


In [14]:
path = src.PATH / "overleaf/tables/summary.tex"

summary_table.to_latex(
    path,
    float_format="{:.1f}".format,
    escape=True,
    column_format="lrrrrrrrr",
)

summary_table.to_csv(path.with_suffix(".csv"), index=True)

NameError: name 'summary_table' is not defined

In [17]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = likes + 1,
      views = views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=2)

ggsave(here("overleaf/img/view_count_violin.pdf"))
ggsave(here("overleaf/img/view_count_violin.svg"))

Saving 13.9 x 8.33 in image
Saving 13.9 x 8.33 in image


In [18]:
%%R -i df -i r_colormap -w 800 -h 1000

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)
df_plot <- df %>%
   mutate(
      likes = likes + 1,
      views = views + 1,
)
like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.7) +
   scale_y_continuous(trans="log10") +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

ggarrange(view_plot, like_plot, ncol=1)

ggsave(here("overleaf/img/view_count_violin_vertical.pdf"))
ggsave(here("overleaf/img/view_count_violin_vertical.svg"))

Saving 11.1 x 13.9 in image
Saving 11.1 x 13.9 in image
